# TD1 Système de recommandation

Calculer la similarité entre les profils de deux clients, entre deux produits,
permet d’identifier les centres d’interet ”proches”. Ainsi, plus la ”distance”
entre deux ”objets” est petite, plus la similarite est grande, i.e. plus ils sont
similaires. Pour realiser une recommandation on peut alors proceder de la sorte:
<br>
•calculer les similarites entre les objets
<br>
•trier les objets dans l’ordre d ́ecroissant
<br>
•retourner les N premiers objets pertinents

In [25]:
import numpy as np
import pandas as pd
import math
np.set_printoptions(precision=2)

## 3. Systèmes de recommandation basiques

### 3.0 Création des données

In [26]:
# Fonction de similarité (Cosinus)
def cosine_sim(vec_a, vec_b):
    dot = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot / (norm_a * norm_b) if norm_a > 0 and norm_b > 0 else 0

### 3.1 User-based

In [27]:
# DataFrames
# Encodé (15->0, 28->1, 45->2, 21->1)
data_profils = {
    'Age':      [1, 2, 3, 2],
    'Sexe':     [1, 0, 1, 1],           
    'Action':   [1, 1, 0, 0],
    'Aventure': [1, 0, 0, 1],
    'SF':       [0, 1, 0, 1],
    'Comedie':  [0, 0, 1, 0],
    'Drame':    [0, 0, 1, 0]
}

# Index
df_users = pd.DataFrame(data_profils, index=['U1', 'U2', 'U3', 'U4'])

print("--- DataFrame des Profils ---")
display(df_users)

# Création du DataFrame des notes(U1, U2, U3 pour M1-M4)
data_notes = {
    'M1': [4, 5, 2],
    'M2': [3, 4, 5],
    'M3': [2, 3, 4],
    'M4': [4, 3, 5]
}
df_notes = pd.DataFrame(data_notes, index=['U1', 'U2', 'U3'])

print("\n--- DataFrame des Notes ---")
display(df_notes)


target_user_id = 'U4'
target_vec = df_users.loc[target_user_id] 
neighbors_df = df_users.drop(target_user_id)

# Calcul des Similarités
# On applique la fonction cosine_sim entre U4 et chaque ligne
similarities = neighbors_df.apply(lambda row: cosine_sim(target_vec, row), axis=1)

print("\n--- Similarités avec U4 ---")
print(similarities)

# Prédiction des notes

# On utilise le produit scalaire (dot) pour faire la somme pondérée
weighted_sum = df_notes.T.dot(similarities)

# Diviser par la somme des similarités
sum_of_sims = similarities.sum()
predicted_ratings = weighted_sum / sum_of_sims

print("\n--- Prédictions pour U4 ---")
print(predicted_ratings.round(2))

# Recommandation

top_2_reco = predicted_ratings.sort_values(ascending=False).head(2)

print(f"\n=> RECOMMANDATION POUR U4 : {top_2_reco.index[0]} et {top_2_reco.index[1]}")

--- DataFrame des Profils ---


,Age,Sexe,Action,Aventure,SF,Comedie,Drame
U1,1,1,1,1,0,0,0
U2,2,0,1,0,1,0,0
U3,3,1,0,0,0,1,1
U4,2,1,0,1,1,0,0



--- DataFrame des Notes ---


,M1,M2,M3,M4
U1,4,3,2,4
U2,5,4,3,3
U3,2,5,4,5



--- Similarités avec U4 ---
U1    0.755929
U2    0.771517
U3    0.763763
dtype: float64

--- Prédictions pour U4 ---
M1    3.67
M2    4.00
M3    3.00
M4    4.00
dtype: float64

=> RECOMMANDATION POUR U4 : M2 et M4


### 3.2 Recommandation Item-based

In [ ]:
# DataFrames

features = ['Classification', 'Action', 'Aventure', 'SF', 'Comedie', 'Drame']
# Films déjà notés par U4 (M1 à M4)
data_rated = np.array([
    [1, 1, 0, 0, 0, 0],
    [2, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 1, 0],
    [3, 0, 0, 0, 0, 1]
])
df_rated = pd.DataFrame(data_rated, columns=features, index=['M1', 'M2', 'M3', 'M4'])

# Films à prédire (M5 à M8)
data_target = np.array([
    [0, 1, 0, 0, 0, 0],
    [4, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0]
])
df_target = pd.DataFrame(data_target, columns=features, index=['M5', 'M6', 'M7', 'M8'])

# Notes de U4
u4_ratings = pd.Series([4, 3, 1, 5], index=['M1', 'M2', 'M3', 'M4'])

print("--- Films déjà notés par U4 ---")
display(df_rated)
print("\n--- Films à prédire ---")
display(df_target)

# Matrice de Similarité

# Fonction pour calculer la sim d'un film cible (row) contre les films notés (df_rated)
def calculate_sims(target_row):
    return df_rated.apply(lambda rated_row: cosine_sim(target_row, rated_row), axis=1)

# On applique cette fonction à chaque ligne de df_target
sim_matrix = df_target.apply(calculate_sims, axis=1)

print("\n--- Matrice de Similarité (Cible vs Historique) ---")
display(sim_matrix.round(3))

# Prédiction

numerator = sim_matrix.dot(u4_ratings)
denominator = sim_matrix.sum(axis=1) # Somme des similarités par ligne

# Division et gestion du cas division par zéro
pred_ratings = numerator.divide(denominator).fillna(0)

print("\n--- Résultats ---")
print(pred_ratings.round(2))

# Recommandation
top_2 = pred_ratings.sort_values(ascending=False).head(2)
print(f"\n=> RECOMMANDATION POUR U4 (Basé sur contenu) : {top_2.index[0]} et {top_2.index[1]}")